# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Role of TF-IDF
This is the seventh stage of the pipeline. Machine Learning algorithms cannot understand raw text. We use TF-IDF (Term Frequency - Inverse Document Frequency) to convert the cleaned text from the previous stage into sparse numerical matrices that represent the relative importance of words and phrases.

## Critical Data Leakage Rule
The TF-IDF vectorizer must **ONLY be fitted on the training data**. If we fit it on the entire dataset, the vocabulary and inverse document frequencies will be influenced by the validation and test sets, artificially boosting model performance and leading to invalid research results.


# 2. TF-IDF OBJECTIVES

Our goals for this stage are to:
- Convert processed Indonesian text into numerical features using TF-IDF.
- Capture single words and short phrases using appropriate N-Grams.
- Produce highly efficient sparse feature matrices.
- Strictly prevent data leakage by splitting the dataset before vectorization.
- Generate reusable TF-IDF artifacts for Machine Learning training.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import json
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import warnings

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [13]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TFIDF_OUTPUT_DIR = DATA_PROCESSED_DIR / "tfidf"
MODELS_DIR = PROJECT_ROOT / "models"
BASE_REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR = BASE_REPORTS_DIR / "tfidf"

PROCESSED_DATA_PATH = DATA_PROCESSED_DIR / "processed_cyberbullying_dataset.csv"

# Directories creation
TFIDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Columns
TEXT_COL = "text_processed"
LABEL_COL = "cyberbullying_type"

# Split Config
RANDOM_STATE = 42
TEST_SIZE_INITIAL = 0.30  # 70% Train, 30% for Val+Test
TEST_SIZE_FINAL = 0.50    # Split the 30% into 15% Val, 15% Test

# TF-IDF Config
TFIDF_PARAMS = {
    "ngram_range": (1, 4),    # Extended to Trigrams
    "min_df": 1,              # Ignore terms appearing in less than 5 docs
    "max_df": 0.9,            # Ignore terms appearing in >90% of docs (implicit stopwords)
    "sublinear_tf": True      # Logarithmic scaling of term frequency
}

print(f"Processed Data Path: {PROCESSED_DATA_PATH}")


Processed Data Path: /home/zapp/Kampus/PM-NEW/data/processed/processed_cyberbullying_dataset.csv


In [3]:
# 5. LOAD PROCESSED DATASET

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"FAIL: Processed dataset not found at {PROCESSED_DATA_PATH}. Please run 05_preprocessing.ipynb first.")

df = pd.read_csv(PROCESSED_DATA_PATH)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
display(df.head())


Dataset Shape: 43137 rows, 9 columns
Columns: ['text', 'text_processed', 'cyberbullying_type', 'severity_level', 'original_cyberbullying_type', 'original_severity_level', 'source_dataset', 'source_file', 'original_row_id']


,text,text_processed,cyberbullying_type,severity_level,original_cyberbullying_type,original_severity_level,source_dataset,source_file,original_row_id
0,setiap orang adalah seorang gadis yang akan me...,setiap orang adalah seorang gadis yang akan me...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,0
1,bahwa pos ab kpop stans pergi ke sekolah bersa...,bahwa pos ab kpop stans pergi ke sekolah bersa...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,1
2,karena beberapa orang tidak ada yang lebih bai...,karena beberapa orang tidak ada yang lebih bai...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,2
3,bro aku pelatih jv tahun lalu di skyline dan a...,bro aku pelatih jv tahun lalu di skyline dan a...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,3
4,wanitawanita ini benarbenar mengingatkan saya ...,wanitawanita ini benarbenar mengingatkan saya ...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,4


In [4]:
# 6. VERIFY DATASET STRUCTURE

required_cols = [TEXT_COL, LABEL_COL]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"CRITICAL ERROR: Dataset is missing required columns: {missing_cols}")

# Ensure text column doesn't have NaN (which might have slipped if saved incorrectly)
if df[TEXT_COL].isnull().sum() > 0:
    print(f"WARNING: Found {df[TEXT_COL].isnull().sum()} null values in processed text. Filling with empty string.")
    df[TEXT_COL] = df[TEXT_COL].fillna("")

print("Input dataset verified.")


Input dataset verified.


In [5]:
# 7. DEFINE FEATURES AND TARGETS

X = df[TEXT_COL]
y = df[LABEL_COL]

# We also preserve the entire dataframe rows for metadata extraction later
metadata_cols = [c for c in df.columns if c not in [TEXT_COL, LABEL_COL]]

print(f"Features (X) shape: {X.shape}")
print(f"Targets (y) shape: {y.shape}")


Features (X) shape: (43137,)
Targets (y) shape: (43137,)


In [6]:
# 8. TRAIN-VALIDATION-TEST SPLIT

# 1. Split Train (70%) and Temp (30%)
# Stratification ensures class distribution remains consistent across splits.
X_train, X_temp, y_train, y_temp = train_test_split(
    df, y, 
    test_size=TEST_SIZE_INITIAL, 
    random_state=RANDOM_STATE, 
    stratify=y
)

# 2. Split Temp (30%) into Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=TEST_SIZE_FINAL, 
    random_state=RANDOM_STATE, 
    stratify=y_temp
)

print(f"Train size: {len(X_train)} ({(len(X_train)/len(df))*100:.1f}%)")
print(f"Validation size: {len(X_val)} ({(len(X_val)/len(df))*100:.1f}%)")
print(f"Test size: {len(X_test)} ({(len(X_test)/len(df))*100:.1f}%)")


Train size: 30195 (70.0%)
Validation size: 6471 (15.0%)
Test size: 6471 (15.0%)


In [7]:
# 9. VALIDATE SPLIT DISTRIBUTION

dist_df = pd.DataFrame({
    'Train': y_train.value_counts(normalize=True) * 100,
    'Validation': y_val.value_counts(normalize=True) * 100,
    'Test': y_test.value_counts(normalize=True) * 100
}).round(2)

print("Class Distribution Percentages Across Splits:")
display(dist_df)


Class Distribution Percentages Across Splits:


,Train,Validation,Test
cyberbullying_type,,,
normal,66.87,66.88,66.87
hate_speech,16.97,16.97,16.98
harassment,12.12,12.12,12.12
insult,4.04,4.03,4.03


In [8]:
# 10. DEFINE TF-IDF VECTORIZER

vectorizer = TfidfVectorizer(
    ngram_range=TFIDF_PARAMS['ngram_range'],
    min_df=TFIDF_PARAMS['min_df'],
    max_df=TFIDF_PARAMS['max_df'],
    sublinear_tf=TFIDF_PARAMS['sublinear_tf']
)

print(f"Vectorizer defined with parameters: {TFIDF_PARAMS}")


Vectorizer defined with parameters: {'ngram_range': (1, 3), 'min_df': 2, 'max_df': 0.9, 'sublinear_tf': True}


In [9]:
# 11. FIT TF-IDF ON TRAINING DATA

# Extract only the text column
X_train_text = X_train[TEXT_COL]

print("Fitting TF-IDF on Training Data ONLY...")
X_train_tfidf = vectorizer.fit_transform(X_train_text)

print(f"Training TF-IDF Matrix Shape: {X_train_tfidf.shape}")


Fitting TF-IDF on Training Data ONLY...
Training TF-IDF Matrix Shape: (30195, 207616)


In [10]:
# 12. TRANSFORM VALIDATION AND TEST DATA

X_val_text = X_val[TEXT_COL]
X_test_text = X_test[TEXT_COL]

print("Transforming Validation and Test Data...")
X_val_tfidf = vectorizer.transform(X_val_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print(f"Validation TF-IDF Matrix Shape: {X_val_tfidf.shape}")
print(f"Test TF-IDF Matrix Shape: {X_test_tfidf.shape}")


Transforming Validation and Test Data...
Validation TF-IDF Matrix Shape: (6471, 207616)
Test TF-IDF Matrix Shape: (6471, 207616)


In [11]:
# 13. VALIDATE FEATURE MATRICES

assert X_train_tfidf.shape[1] == X_val_tfidf.shape[1], "Dimension mismatch between Train and Validation!"
assert X_val_tfidf.shape[1] == X_test_tfidf.shape[1], "Dimension mismatch between Validation and Test!"
assert X_train_tfidf.shape[0] == len(y_train), "Row mismatch between X_train and y_train!"

print("Feature matrices dimensions are consistent and validated.")


Feature matrices dimensions are consistent and validated.


In [12]:
# 14. ANALYZE VOCABULARY

vocab_size = len(vectorizer.vocabulary_)
feature_names = vectorizer.get_feature_names_out()

print(f"Total Vocabulary Size (Features): {vocab_size}")
print(f"\nSample features (first 10): {feature_names[:10]}")
print(f"Sample features (random 10): {np.random.choice(feature_names, 10)}")


Total Vocabulary Size (Features): 207616

Sample features (first 10): ['00' '00 11' '00 11 00' '00 11 20' '00 11 88' '00 12' '00 12 30' '00 13'
 '00 13 00' '00 14']
Sample features (random 10): ['dan hakhak' 'diamankan satpol' 'jakarta dari' 'epistemologi'
 'pemboman yg' 'bio photo freepik' 'sudah ada satu' 'gara roy'
 'yg sama juga' 'ios 17']


In [14]:
# 15. ANALYZE TF-IDF FEATURES

def analyze_sparse_matrix(name, matrix):
    non_zero = matrix.nnz
    total_elements = matrix.shape[0] * matrix.shape[1]
    sparsity = 1.0 - (non_zero / total_elements)
    
    print(f"--- {name} ---")
    print(f"Shape: {matrix.shape}")
    print(f"Non-zero elements: {non_zero}")
    print(f"Sparsity: {sparsity*100:.4f}% ({(1-sparsity)*100:.4f}% dense)\n")

analyze_sparse_matrix("Training Matrix", X_train_tfidf)
analyze_sparse_matrix("Validation Matrix", X_val_tfidf)
analyze_sparse_matrix("Test Matrix", X_test_tfidf)


--- Training Matrix ---
Shape: (30195, 207616)
Non-zero elements: 1400870
Sparsity: 99.9777% (0.0223% dense)

--- Validation Matrix ---
Shape: (6471, 207616)
Non-zero elements: 243812
Sparsity: 99.9819% (0.0181% dense)

--- Test Matrix ---
Shape: (6471, 207616)
Non-zero elements: 243474
Sparsity: 99.9819% (0.0181% dense)



In [15]:
# 16. ANALYZE TOP TF-IDF TERMS

# We calculate the average TF-IDF score for each feature across all training documents
average_tfidf_scores = X_train_tfidf.mean(axis=0).A1
top_indices = average_tfidf_scores.argsort()[::-1][:20]
top_features = [(feature_names[i], average_tfidf_scores[i]) for i in top_indices]

print("Top 20 Features with Highest Average TF-IDF Weight in Training Data:")
for rank, (feature, score) in enumerate(top_features, 1):
    print(f"{rank:2d}. {feature:<20} : {score:.4f}")
    
print("\nNote: High average TF-IDF weight does not necessarily mean the term is the most predictive feature for the final classification model.")


Top 20 Features with Highest Average TF-IDF Weight in Training Data:
 1. user                 : 0.0220
 2. di                   : 0.0155
 3. dan                  : 0.0130
 4. yang                 : 0.0124
 5. israel               : 0.0122
 6. gila                 : 0.0115
 7. user user            : 0.0112
 8. ini                  : 0.0105
 9. yg                   : 0.0095
10. china                : 0.0078
11. itu                  : 0.0076
12. dari                 : 0.0070
13. untuk                : 0.0069
14. orang                : 0.0069
15. user user user       : 0.0068
16. ada                  : 0.0067
17. user_mention         : 0.0065
18. ke                   : 0.0065
19. dengan               : 0.0062
20. indonesia            : 0.0062

Note: High average TF-IDF weight does not necessarily mean the term is the most predictive feature for the final classification model.


# 17. VALIDATE DATA LEAKAGE PREVENTION

**Data Leakage Prevention Validation:**
- **Fit:** The `vectorizer.fit_transform()` was explicitly called *only* on `X_train_text` (Cell 11).
- **Transform:** The `vectorizer.transform()` method was strictly used for `X_val_text` and `X_test_text` (Cell 12).
- **Vocabulary:** The vocabulary size (`vocab_size`) is entirely determined by the 70% training split. Unseen words in Validation or Test sets are safely ignored.
- The workflow satisfies the strict data leakage requirements.


In [16]:
# 18. SAVE TF-IDF ARTIFACTS

# Save Matrices
sparse.save_npz(TFIDF_OUTPUT_DIR / "X_train_tfidf.npz", X_train_tfidf)
sparse.save_npz(TFIDF_OUTPUT_DIR / "X_val_tfidf.npz", X_val_tfidf)
sparse.save_npz(TFIDF_OUTPUT_DIR / "X_test_tfidf.npz", X_test_tfidf)

# Save Target Labels (along with metadata)
X_train.to_csv(TFIDF_OUTPUT_DIR / "train_metadata.csv", index=False)
X_val.to_csv(TFIDF_OUTPUT_DIR / "val_metadata.csv", index=False)
X_test.to_csv(TFIDF_OUTPUT_DIR / "test_metadata.csv", index=False)

# Keep simple label series as well for easy access
y_train.to_csv(TFIDF_OUTPUT_DIR / "y_train.csv", index=False)
y_val.to_csv(TFIDF_OUTPUT_DIR / "y_val.csv", index=False)
y_test.to_csv(TFIDF_OUTPUT_DIR / "y_test.csv", index=False)

# Save Vectorizer
vectorizer_path = MODELS_DIR / "tfidf_vectorizer.pkl"
joblib.dump(vectorizer, vectorizer_path)

print("All TF-IDF artifacts saved successfully.")


All TF-IDF artifacts saved successfully.


In [17]:
# 19. SAVE METADATA

metadata = {
    "random_state": RANDOM_STATE,
    "train_ratio": 0.7,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "train_rows": int(X_train_tfidf.shape[0]),
    "val_rows": int(X_val_tfidf.shape[0]),
    "test_rows": int(X_test_tfidf.shape[0]),
    "vocabulary_size": int(vocab_size),
    "tfidf_parameters": TFIDF_PARAMS,
    "data_leakage_prevented": True
}

meta_path = REPORTS_DIR / "tfidf_metadata.json"
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"TF-IDF metadata saved to {meta_path}")


TF-IDF metadata saved to /home/zapp/Kampus/PM-NEW/reports/tfidf/tfidf_metadata.json


# 20. TF-IDF SUMMARY

### Input
- **Dataset**: `processed_cyberbullying_dataset.csv`
- **Total Records**: See Split.

### Split (70 / 15 / 15)
- **Train**: Used for fitting vocabulary and weights.
- **Validation**: Used for hyperparameter tuning in later stages.
- **Test**: Kept strictly isolated for final model evaluation.

### TF-IDF Configuration
- `ngram_range`: (1, 2)
- `min_df`: 5
- `max_df`: 0.9
- `sublinear_tf`: True

### Feature Space
- **Vocabulary Size**: Evaluated dynamically based on training data.
- **Sparsity**: Expected to be >99%, saved efficiently in `.npz` format.

### Output Artifacts
- **Matrices**: `X_train_tfidf.npz`, `X_val_tfidf.npz`, `X_test_tfidf.npz`
- **Labels & Metadata**: `y_train.csv`, `train_metadata.csv`, etc.
- **Model**: `tfidf_vectorizer.pkl`

### Next Step
The numerical feature matrices are ready. Proceed to `notebooks/07_model_training.ipynb` to begin training classical ML models.


# How to Run This Notebook

1. Activate the project's virtual environment.
2. Ensure the processed dataset exists in: `data/processed/processed_cyberbullying_dataset.csv`
3. Ensure the preprocessing stage has been completed: `notebooks/05_preprocessing.ipynb`
4. Open: `notebooks/06_tfidf.ipynb`
5. Verify the configured paths in Step 4.
6. Review the train-validation-test split configuration (Step 8) and TF-IDF config (Step 10).
7. Run the notebook from the first cell to the last cell.
8. Review the split distributions to ensure balanced stratification (Step 9).
9. Review the TF-IDF feature dimensions and sparsity (Step 13, 15).
10. Confirm that the artifacts were saved in `data/processed/tfidf/` and `models/`.
11. Proceed to: `notebooks/07_model_training.ipynb`
